# 🥊 Bangla TTS Shootout — promito Bangla er jonno best model khuji

Compare kori (eki Bangladeshi promito reference + eki text-e):
1. **base IndicF5** (ai4bharat/IndicF5) — strong Bengali phonology, Indian-leaning
2. **ehzawad/indicf5-bangla-tts** — IndicF5, **Bangladeshi Bengali** e fine-tuned

Duitai tomar **BD promito reference** clone kore। Shune je-ta best **Bangladeshi promito**, oita app-e boshabo।

**Age koro:** Runtime → Change runtime type → **T4 GPU** → Save.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU nai! Runtime > Change runtime type > T4 GPU koro.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Install IndicF5 (~5 min)
> pip-e alada `f5_tts` thakle IndicF5 er vendored version shadow kore — uninstall kori.

In [ ]:
!pip -q uninstall -y f5_tts f5-tts 2>/dev/null
!pip -q install git+https://github.com/ai4bharat/IndicF5.git
!pip -q install faster-whisper soundfile yt-dlp
print('Installed \u2713  (kono dependency ERROR thakle amake dekhao)')

## 3. Bangladeshi promito reference
**REF_YT** e ekta clean **Bangladeshi promito** YouTube link dao (news bulletin / clear speaker)। REF_START theke REF_DUR (sec) segment nibe।

(Chaile REF_YT khali rekhe niche upload-o kora jay।)

In [ ]:
REF_YT = ""          # <-- Bangladeshi promito YouTube link ekhane
REF_START = 5          # kototomo second theke
REF_DUR = 12           # koto second (8-15 valo)

import os
os.makedirs('/content/out', exist_ok=True)
REF = '/content/ref.wav'

if REF_YT.strip():
    seg = f"*{REF_START}-{REF_START + REF_DUR}"
    !yt-dlp -f bestaudio/best --no-playlist --quiet --no-warnings --download-sections "{seg}" -o /content/ref_src.%(ext)s "{REF_YT}"
    import glob
    s = glob.glob('/content/ref_src.*')
    if not s:
        !yt-dlp -f bestaudio/best --no-playlist --quiet --no-warnings -o /content/ref_src.%(ext)s "{REF_YT}"
        s = glob.glob('/content/ref_src.*')
    !ffmpeg -y -ss {REF_START} -t {REF_DUR} -i "{s[0]}" -ac 1 -ar 24000 {REF} -loglevel error
else:
    from google.colab import files
    print('REF_YT khali — file upload koro (Bangladeshi promito clip):')
    up = files.upload()
    !ffmpeg -y -i "{list(up.keys())[0]}" -ac 1 -ar 24000 -t 15 {REF} -loglevel error

print('Reference ready:', REF)
from IPython.display import Audio; Audio(REF)

## 4. Reference-er transcript (IndicF5 er dorkar)
Auto-transcribe (faster-whisper, bn)। `REF_TEXT` check koro — vul thakle niche hate thik koro।

In [ ]:
from faster_whisper import WhisperModel
wm = WhisperModel('medium', device='cuda', compute_type='int8_float16')
segs, _ = wm.transcribe(REF, language='bn', beam_size=5)
REF_TEXT = ''.join(s.text for s in segs).strip()
del wm; import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('REF_TEXT =', REF_TEXT)

In [ ]:
# Vul thakle uncomment kore hate thik koro:
# REF_TEXT = "reference clip e thik ja bola hoyeche"
print('Using REF_TEXT:', REF_TEXT)

## 5. Target promito Bangla text (ja bolabe)

In [ ]:
TARGET = "সুপ্রভাত। আজকের আবহাওয়া সম্পর্কে কিছু গুরুত্বপূর্ণ তথ্য জানানো হচ্ছে। তাপমাত্রা ছত্রিশ ডিগ্রি সেলসিয়াস পর্যন্ত উঠতে পারে।"
print(TARGET)

## 6. \ud83c\udd70\ufe0f base IndicF5 (ai4bharat)

In [ ]:
import numpy as np, soundfile as sf
from transformers import AutoModel

def to_f32(a):
    a = np.asarray(a)
    if a.dtype == np.int16:
        a = a.astype(np.float32) / 32768.0
    return a.astype(np.float32).squeeze()

m_base = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True).to('cuda')
wav = m_base(TARGET, ref_audio_path=REF, ref_text=REF_TEXT)
sf.write('/content/out/A_indicf5_base.wav', to_f32(wav), 24000)
print('Saved: A_indicf5_base.wav')
from IPython.display import Audio; Audio('/content/out/A_indicf5_base.wav')

## 7. \ud83c\udd71\ufe0f ehzawad/indicf5-bangla-tts (Bangladeshi fine-tune)

In [ ]:
del m_base; import gc, torch; gc.collect(); torch.cuda.empty_cache()
m_bd = AutoModel.from_pretrained('ehzawad/indicf5-bangla-tts', trust_remote_code=True).to('cuda')
wav = m_bd(TARGET, ref_audio_path=REF, ref_text=REF_TEXT)
sf.write('/content/out/B_indicf5_bangladeshi.wav', to_f32(wav), 24000)
print('Saved: B_indicf5_bangladeshi.wav')
from IPython.display import Audio; Audio('/content/out/B_indicf5_bangladeshi.wav')

## 8. Compare + download
**A** (base IndicF5) vs **B** (Bangladeshi IndicF5) vs app-er **Chatterbox** — kon-ta best Bangladeshi promito? Amake bolo।

In [ ]:
from google.colab import files
for f in ['A_indicf5_base.wav', 'B_indicf5_bangladeshi.wav']:
    files.download('/content/out/' + f)